# Experiment 2 — Seed sweep

Bootstrap intervals in the paper resample the *evaluation set* for a model
whose weights are fixed. They say nothing about how much the result moves when
the same recipe is trained again from a different initialisation. A referee
will ask, and the honest answer today is that we do not know.

This trains two more models on **exactly the data of the original run**,
differing only in `project.seed`, and reports the spread.

The claim to support is undemanding: the S1-to-S5 gap is 0.66, so almost any
plausible seed-to-seed spread leaves it intact. This is cheap insurance rather
than a risk, which is why it is second priority behind the matched-scale run.

## Settings

Same as Experiment 1: `GPU T4 x2`, Internet `On`, and **Save & Run All
(Commit)** rather than interactive.

**Budget: 6-7 hours** (about an hour of data prep, then 2.5 h per seed).

> A bug worth knowing about, now fixed: `project.seed` used to reach only
> pandas sampling. Nothing seeded torch, so weight initialisation, dropout and
> batch order were left to default entropy. Before the fix this experiment
> would have run and produced numbers that varied for reasons unrelated to the
> seed.

In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a pipeline stage and stop the notebook if it fails.

    Without the raise a failed stage prints a traceback and the next cell
    happily trains on whatever stale data is lying around.
    """
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Add your code dataset: right panel -> Input -> Add Input -> Datasets,\n"
        "then search for the dataset you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
print("configs present:", sorted(p.name for p in (dest/"configs").glob("kaggle*.yaml")))

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Corpus

One shard, exactly as the original run, so these seeds are comparable with the
numbers already in the paper.

In [ ]:
run(["-m", "aicd.data.download", "--config", "kaggle.yaml", "--train-shards", "1"])
for stage in ["normalize", "filter", "splits"]:
    run(["-m", f"aicd.data.{stage}", "--config", "kaggle.yaml"])
    elapsed(stage)

run(["-m", "pytest", "aicd/tests/", "-q"])

sp = json.load(open(WORK / "aicd" / "eval" / "reports" / "splits.json"))
print(f"training rows: {sp['train']['rows']:,}  (paper: 196,854)")
if abs(sp["train"]["rows"] - 196854) > 20000:
    print("WARNING: this does not match the published split. These seeds will")
    print("not be directly comparable with Table III.")

## 4. Train the two extra seeds

Each gets its own `--tag`, so the three runs cannot overwrite one another.

`--resume` is safe here *because* the tags differ: a resumed run continues its
own checkpoint. Re-run the cell after an interruption and it continues where it
stopped.

In [ ]:
for seed in [1, 2]:
    run(["-m", "aicd.models.modernbert_triplet",
         "--config", f"kaggle_seed{seed}.yaml",
         "--tag", f"seed{seed}", "--resume"])
    elapsed(f"seed {seed} done")

## 5. Spread across seeds

In [ ]:
import numpy as np
rep = WORK / "aicd" / "eval" / "reports"

runs = {"paper (seed 20260818)": {"s1_in_distribution": 0.8977,
                                  "s5_compound": 0.2378}}
for seed in [1, 2]:
    f = rep / f"branch_a_seed{seed}.json"
    if f.exists():
        r = json.load(open(f))["slices"]
        runs[f"seed {seed}"] = {k: r[k]["macro_f1"] for k in
                                ["s1_in_distribution", "s5_compound"] if k in r}

print(f"{'run':24s} {'S1':>9s} {'S5':>9s}")
print("-" * 45)
for name, v in runs.items():
    print(f"{name:24s} {v.get('s1_in_distribution', float('nan')):9.4f} "
          f"{v.get('s5_compound', float('nan')):9.4f}")

s1 = [v["s1_in_distribution"] for v in runs.values() if "s1_in_distribution" in v]
s5 = [v["s5_compound"] for v in runs.values() if "s5_compound" in v]
if len(s1) > 1:
    print(f"\nS1  mean {np.mean(s1):.4f}  sd {np.std(s1, ddof=1):.4f}")
    print(f"S5  mean {np.mean(s5):.4f}  sd {np.std(s5, ddof=1):.4f}")
    gap = np.mean(s1) - np.mean(s5)
    sd = max(np.std(s1, ddof=1), np.std(s5, ddof=1))
    print(f"\ngap {gap:.4f} vs largest seed sd {sd:.4f}  ->  {gap/max(sd,1e-9):.0f}x")
    print("\nReport as: mean +/- sd over 3 seeds, in Table III.")

## 6. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)

# The probability arrays are what the analysis modules re-read at home, and
# they are small. The model weights are hundreds of MB and are not needed to
# reproduce any number in the paper, so they stay behind.
art = WORK / "aicd" / "artifacts"
npy = OUT / "arrays"
npy.mkdir(exist_ok=True)
n = 0
for f in art.glob("proba_a*.npy"):
    shutil.copy(f, npy / f.name); n += 1
for f in art.glob("labels.parquet"):
    shutil.copy(f, npy / f.name)
if (art / "kaggle").exists():
    for f in (art / "kaggle").glob("*"):
        if f.is_file() and f.stat().st_size < 200e6:
            shutil.copy(f, npy / f.name); n += 1

shutil.make_archive("/kaggle/working/results", "zip", OUT)
print(f"copied {n} arrays")
print("-> /kaggle/working/results.zip  (download this from the Output tab)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")